# SOTA Championship V2: Cross-Modal Additive Angular Margin Pipeline
### WACV 2027 UPAR Challenge & MEVID Benchmark | Target: mADM >= 0.6850 (Live #1: 0.6783)
---
This notebook executes the **SOTA Championship V2 Two-Stage Retraining Pipeline** for Attribute-Based Person Retrieval on Kaggle Dual Tesla T4 GPUs.

### Architectural Innovations:
- **CLEAR Deconstructed Architecture:** Decoupled two-stage training with frozen visual backbone and specialized query adapters.
- **Dual-Query Fusion:** Concatenation of Hard Binary Query (40-D to 512-D) and Soft Contextual Query (CLIP Text to 512-D) into a unified 1024-D representation.
- **Additive Angular Margin Loss (ArcFace):** Enforces angular margin m=0.35 with scale s=30.0 on the unit hypersphere to separate low-resolution surveillance clusters.
- **Anatomical 3-Strip Partitioning:** Head (25%), Torso (40%), Legs (35%) with learnable Generalized Mean (GeM) Pooling.
- **Asymmetric Multi-Label Loss (ASL):** Dynamic positive/negative gamma decoupling (gamma_+=0, gamma_-=4) for severe attribute imbalance.
- **High-Throughput Dual T4 DDP:** Batch size 96/GPU, accumulation 2, effective batch size 384, full AMP FP16, RAM caching for zero disk I/O bottlenecks.
- **Automated Submission Packaging & Isolated Container Audit:** Directly generates verified air-gapped submission_sota_v2.zip.


In [ ]:
# ==============================================================================
# CELL 1: HARDWARE & SYSTEM TELEMETRY CHECK
# ==============================================================================
import os
import sys
import psutil
import subprocess
import torch

print("=" * 80)
print("[HARDWARE] Inspecting Dual Tesla T4 Accelerators & Host Environment...")
print("=" * 80)

cuda_available = torch.cuda.is_available()
gpu_count = torch.cuda.device_count() if cuda_available else 0
print(f"CUDA Available: {cuda_available} | Device Count: {gpu_count}")

for i in range(gpu_count):
    props = torch.cuda.get_device_properties(i)
    print(f"  -> GPU {i}: {props.name} | Total VRAM: {props.total_memory / (1024**3):.2f} GB")

vmem = psutil.virtual_memory()
print(f"Host CPU Cores: {os.cpu_count()} | Total RAM: {vmem.total / (1024**3):.2f} GB | Available: {vmem.available / (1024**3):.2f} GB")
print("=" * 80)

# Install required SOTA vision-language dependencies
print("[DEPS] Installing OpenCLIP, Timm, and Accelerate dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "open-clip-torch", "timm", "accelerate"], check=True)
print("[DEPS] Dependencies installed successfully!")


In [ ]:
# ==============================================================================
# CELL 2: CLONE OR PULL PRIVATE REPOSITORY VIA GITHUB_TOKEN
# ==============================================================================
import os
import sys
import subprocess

REPO_NAME = "upar_challenge_26_rws27"
REPO_DIR = os.path.join("/kaggle/working", REPO_NAME)

token = None
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    token = user_secrets.get_secret("GITHUB_TOKEN")
    print("[AUTH] Successfully retrieved GITHUB_TOKEN from Kaggle Secrets!")
except Exception as e:
    print(f"[AUTH] Note: Kaggle Secrets not accessible ({e}). Attempting public clone...")

if token:
    repo_url = f"https://{token}@github.com/Aditya-Lingam-9000/{REPO_NAME}.git"
else:
    repo_url = f"https://github.com/Aditya-Lingam-9000/{REPO_NAME}.git"

if not os.path.isdir(REPO_DIR):
    print(f"[GIT] Cloning repository into {REPO_DIR}...")
    res = subprocess.run(["git", "clone", repo_url, REPO_DIR], capture_output=True, text=True)
    if res.returncode != 0:
        print(f"[GIT ERROR] Clone failed:\n{res.stderr}")
        raise RuntimeError("Failed to clone repository. Verify GITHUB_TOKEN.")
    print("[GIT] Successfully cloned repository!")
else:
    print(f"[GIT] Repository already exists at {REPO_DIR}. Fetching latest commits from main...")
    res = subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], capture_output=True, text=True)
    print(res.stdout)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f"[SYSTEM] Working directory set to: {os.getcwd()}")


In [ ]:
# ==============================================================================
# CELL 3: MASTER SOTA V2 CONFIGURATION PANEL
# ==============================================================================
import os
import sys
from sota_pipeline.config import SOTAConfig

# Training Configuration (CLEAR Two-Stage Pipeline)
EPOCHS_STAGE1 = 10           # Stage 1: Representation fine-tuning (backbone + adapters)
EPOCHS_STAGE2 = 15           # Stage 2: Frozen backbone + ArcFace additive angular margin learning
BATCH_SIZE_PER_GPU = 96      # 96 per GPU * 2 GPUs = Global: 192 (VRAM: ~13.8 GB / GPU)
ACCUMULATION_STEPS = 2       # Effective global batch size = 384
BACKBONE_LR = 5e-5           # Learning rate for visual backbone
ADAPTER_LR = 3.5e-4          # Learning rate for Hard & Soft query adapters
MIN_LR = 1e-6                # Minimum cosine annealing learning rate floor
ARCFACE_SCALE = 30.0         # ArcFace hyperspherical radius s
ARCFACE_MARGIN = 0.35        # ArcFace angular margin penalty m (rad)
EVAL_EVERY = 1               # Run validation retrieval benchmark after EVERY epoch
VAL_QUERIES = 500            # Number of validation queries for exact CodaBench benchmarking
EVAL_BATCH_SIZE = 384        # Fast validation batch size (<25s per evaluation)
CHECKPOINT_DIR = "checkpoints_sota_v2"
OUTPUT_DIR = "sota_v2_outputs"

# Optional Warmstart: If you have existing trained weights from Stage 1
# Set WARMSTART_CHECKPOINT = "checkpoints_sota/best_model.pt" to skip directly to Stage 2 ArcFace
WARMSTART_CHECKPOINT = None
prev_best = "checkpoints_sota/best_model.pt"
if os.path.isfile(prev_best):
    print(f"[WARMSTART DETECTED] Found existing Stage 1 checkpoint: {prev_best}")
    # Uncomment below to skip Stage 1 and immediately train Stage 2 ArcFace:
    # WARMSTART_CHECKPOINT = prev_best

# Auto-detect UPAR dataset root
DATASET_ROOT = SOTAConfig.resolve_dataset_root()

print("=" * 80)
print(f"[CONFIG] SOTA Championship V2 Configuration:")
print(f"  - Stage 1 Epochs: {EPOCHS_STAGE1} (Warmstart: {WARMSTART_CHECKPOINT is not None})")
print(f"  - Stage 2 Epochs: {EPOCHS_STAGE2} (ArcFace s={ARCFACE_SCALE}, m={ARCFACE_MARGIN})")
print(f"  - Batch Size: {BATCH_SIZE_PER_GPU}/GPU (Global: {BATCH_SIZE_PER_GPU * 2}, Effective: {BATCH_SIZE_PER_GPU * 2 * ACCUMULATION_STEPS})")
print(f"  - Backbone LR: {BACKBONE_LR} | Adapter LR: {ADAPTER_LR} | Min LR: {MIN_LR}")
print(f"  - Dataset Root: {DATASET_ROOT}")
print(f"  - Checkpoint Dir: {os.path.abspath(CHECKPOINT_DIR)}")
print("=" * 80)


In [ ]:
# ==============================================================================
# CELL 4: HUGGINGFACE ACCELERATE DUAL T4 MULTI-GPU CONFIGURATION
# ==============================================================================
import os
import yaml

accelerate_config = {
    "compute_environment": "LOCAL_MACHINE",
    "distributed_type": "MULTI_GPU",
    "num_machines": 1,
    "num_processes": 2,
    "gpu_ids": "all",
    "mixed_precision": "fp16",
    "main_process_port": 29500,
    "rdzv_backend": "static",
    "same_network": True,
}

config_path = "accelerate_config.yaml"
with open(config_path, "w") as f:
    yaml.dump(accelerate_config, f, default_flow_style=False)

print(f"[ACCELERATE] Generated multi-GPU configuration at: {os.path.abspath(config_path)}")


In [ ]:
# ==============================================================================
# CELL 5: LAUNCH SOTA V2 CHAMPIONSHIP DISTRIBUTED TRAINING
# ==============================================================================
import os
import sys
import shutil
import subprocess

if shutil.which("accelerate"):
    launcher = ["accelerate", "launch", "--config_file", "accelerate_config.yaml"]
else:
    launcher = ["torchrun", "--nproc_per_node=2"]

cmd = launcher + [
    "-m", "sota_pipeline.train_v2",
    f"--epochs_stage1={EPOCHS_STAGE1}",
    f"--epochs_stage2={EPOCHS_STAGE2}",
    f"--batch_size={BATCH_SIZE_PER_GPU}",
    f"--accumulation_steps={ACCUMULATION_STEPS}",
    f"--backbone_lr={BACKBONE_LR}",
    f"--adapter_lr={ADAPTER_LR}",
    f"--min_lr={MIN_LR}",
    f"--arcface_scale={ARCFACE_SCALE}",
    f"--arcface_margin={ARCFACE_MARGIN}",
    f"--dataset_root={DATASET_ROOT}",
    f"--checkpoint_dir={CHECKPOINT_DIR}",
    f"--output_dir={OUTPUT_DIR}",
    f"--eval_every={EVAL_EVERY}",
    f"--val_queries={VAL_QUERIES}",
    f"--eval_batch_size={EVAL_BATCH_SIZE}",
]

if WARMSTART_CHECKPOINT:
    cmd.append(f"--warmstart_checkpoint={WARMSTART_CHECKPOINT}")

print("[EXEC] Executing command:", " ".join(cmd))
print("=" * 80)
subprocess.run(cmd, check=True)


In [ ]:
# ==============================================================================
# CELL 6: POST-TRAINING CHECKPOINT INTEGRITY VERIFICATION
# ==============================================================================
import os
import torch

print("=" * 80)
print("[VERIFY] Auditing Saved SOTA V2 Model Checkpoints...")
print("=" * 80)

best_path = os.path.join(CHECKPOINT_DIR, "best_model.pt")
last_path = os.path.join(CHECKPOINT_DIR, "last_checkpoint.pt")

assert os.path.isfile(best_path), f"Best model checkpoint not found at {best_path}!"
ckpt = torch.load(best_path, map_location="cpu", weights_only=False)
metrics = ckpt.get("metrics", {})
epoch = ckpt.get("epoch", "Unknown")
stage = ckpt.get("stage", "Unknown")
sd = ckpt.get("model_state_dict", ckpt)

print(f"Best Checkpoint: {best_path} ({os.path.getsize(best_path) / (1024**2):.2f} MB)")
print(f"  - Saved at Epoch: {epoch} (Stage {stage})")
print(f"  - Total Parameter Tensors: {len(sd):,}")
print(f"  - Best Validation Metrics:")
print(f"      * mADM:   {metrics.get('mADM', 0.0):.4f} (Live #1: 0.6783)")
print(f"      * Rank-1: {metrics.get('Rank-1', 0.0):.2f}% (Live #1: 40.79%)")
print(f"      * mAP:    {metrics.get('mAP', 0.0):.4f} (Live #1: 0.3952)")
print(f"      * Rank-5: {metrics.get('Rank-5', 0.0):.2f}%")
print(f"      * Rank-10: {metrics.get('Rank-10', 0.0):.2f}%")
print(f"      * mINP:   {metrics.get('mINP', 0.0):.4f}")
print("=" * 80)


In [ ]:
# ==============================================================================
# CELL 7: AUTOMATED SOTA V2 SUBMISSION ZIP PACKAGING
# ==============================================================================
import os
import sys
import subprocess

ZIP_DEST = "/kaggle/working/submission_sota_v2.zip"
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "best_model.pt")
TEMPLATE_DIR = "sota_pipeline/submission_template_v2"

print("=" * 80)
print("[PACKAGING] Exporting Trained Model and Generating CodaBench ZIP...")
print("=" * 80)

export_cmd = [
    sys.executable, "sota_pipeline/export_submission_v2.py",
    f"--checkpoint={CHECKPOINT_PATH}",
    f"--template_dir={TEMPLATE_DIR}",
    f"--output_zip={ZIP_DEST}",
]
subprocess.run(export_cmd, check=True)

zip_size_mb = os.path.getsize(ZIP_DEST) / (1024 * 1024)
print(f"\n[PACKAGE COMPLETE] Generated: {ZIP_DEST} ({zip_size_mb:.2f} MB)")
print("=" * 80)


In [ ]:
# ==============================================================================
# CELL 8: ISOLATED SANDBOX CONTAINER STRUCTURAL AUDIT & RUNTIME TEST
# ==============================================================================
import os
import sys
import ast
import shutil
import zipfile
import tempfile
import subprocess
import torch

ZIP_PATH = "/kaggle/working/submission_sota_v2.zip"
assert os.path.isfile(ZIP_PATH), f"ZIP not found at {ZIP_PATH}!"

SANDBOX_DIR = tempfile.mkdtemp(prefix="codabench_v2_sandbox_audit_")
try:
    print("=" * 80)
    print(f"[SANDBOX AUDIT] Extracting {ZIP_PATH} into clean sandbox: {SANDBOX_DIR}...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(SANDBOX_DIR)

    audit_checks = []

    # 1. run.py existence
    r_path = os.path.join(SANDBOX_DIR, "run.py")
    c1 = os.path.isfile(r_path) and os.path.getsize(r_path) > 10000
    audit_checks.append(("run.py existence and integrity", c1, f"{os.path.getsize(r_path):,} bytes"))

    # 2. metadata.yaml existence
    m_path = os.path.join(SANDBOX_DIR, "metadata.yaml")
    c2 = os.path.isfile(m_path)
    audit_checks.append(("metadata.yaml descriptor presence", c2, "Found"))

    # 3. Model weights existence & size
    w_path = os.path.join(SANDBOX_DIR, "assets", "model_sota_fp16.pt")
    w_mb = os.path.getsize(w_path) / (1024**2) if os.path.isfile(w_path) else 0
    c3 = os.path.isfile(w_path) and w_mb > 200
    audit_checks.append(("model_sota_fp16.pt (> 200 MB)", c3, f"{w_mb:.2f} MB"))

    # 4. Standalone BPE vocabulary existence
    b_path = os.path.join(SANDBOX_DIR, "assets", "bpe_simple_vocab_16e6.txt.gz")
    c4 = os.path.isfile(b_path) and os.path.getsize(b_path) > 1024 * 1024
    audit_checks.append(("bpe_simple_vocab_16e6.txt.gz (> 1 MB)", c4, "Found"))

    # 5. Vendored timm module for zero-internet execution
    v_path = os.path.join(SANDBOX_DIR, "vendor", "timm")
    c5 = os.path.isdir(v_path) and os.path.isfile(os.path.join(v_path, "__init__.py"))
    audit_checks.append(("vendor/timm air-gapped module", c5, "Directory verified"))

    # 6. AST syntax parse & CodaBench entrypoints check
    with open(r_path, "r", encoding="utf-8") as f:
        code_text = f.read()
    parsed_tree = ast.parse(code_text)
    func_names = [n.name for n in ast.walk(parsed_tree) if isinstance(n, ast.FunctionDef)]
    c6 = all(fn in func_names for fn in ["load_model", "rank_gallery"])
    audit_checks.append(("CodaBench entrypoints (load_model, rank_gallery)", c6, "Both confirmed"))

    # 7. Air-gapped code hygiene (no hardcoded absolute machine paths)
    c7 = ("/kaggle/" not in code_text) and ("/Users/" not in code_text) and ("/home/" not in code_text)
    audit_checks.append(("Air-gapped code hygiene (no hardcoded host paths)", c7, "Clean"))

    # 8. Structural weight dictionary inspection (all 4 components present)
    sd = torch.load(w_path, map_location="cpu", weights_only=False)
    has_vis = any(k.startswith("visual_stream.") for k in sd)
    has_txt = any(k.startswith("text_stream.") for k in sd)
    has_hard = any(k.startswith("hard_query_adapter.") for k in sd)
    has_soft = any(k.startswith("soft_query_adapter.") for k in sd)
    all_fp16 = all(v.dtype == torch.float16 for v in sd.values() if isinstance(v, torch.Tensor) and v.is_floating_point())
    has_nan = any(torch.isnan(v).any().item() for v in sd.values() if isinstance(v, torch.Tensor))
    has_inf = any(torch.isinf(v).any().item() for v in sd.values() if isinstance(v, torch.Tensor))
    c8 = has_vis and (not has_nan) and (not has_inf) and all_fp16
    audit_checks.append(("FP16 weight tensor validity (0 NaNs, 0 Infs)", c8, f"FP16: {all_fp16}"))

    # 9. Archive size compliance
    zip_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
    c9 = zip_mb < 500.0
    audit_checks.append(("Archive size compliance (< 500 MB)", c9, f"{zip_mb:.2f} MB"))

    # 10. Air-gapped runtime execution test inside sandbox
    res = subprocess.run([sys.executable, r_path], cwd=SANDBOX_DIR, capture_output=True, text=True)
    c10 = res.returncode == 0 and "Self-test SUCCESS!" in res.stdout
    audit_checks.append(("In-situ air-gapped test execution", c10, "Self-test SUCCESS!"))

    print("=" * 80)
    print("ISOLATED SANDBOX CONTAINER STRUCTURAL AUDIT SCORECARD")
    print("=" * 80)
    all_passed = True
    for name, passed, detail in audit_checks:
        status = "[PASS]" if passed else "[FAIL]"
        print(f"  {status:7s} | {name:<50s} | {detail}")
        if not passed:
            all_passed = False
    print("=" * 80)
    if all_passed:
        print("[AUDIT VERDICT] ALL 10/10 CHECKS PASSED WITH FLYING COLORS!")
        print("  -> Package is 100% self-contained, air-gapped, and competition-ready.")
        print(f"  -> Download '/kaggle/working/submission_sota_v2.zip' and submit to CodaBench!")
    else:
        print("[AUDIT VERDICT] AUDIT FAILED! Review failing checks above.")
        raise RuntimeError("Structural audit failed! Do not submit this archive.")
    print("=" * 80)
finally:
    if os.path.exists(SANDBOX_DIR):
        shutil.rmtree(SANDBOX_DIR)
        print(f"[CLEANUP] Successfully removed sandbox directory: {SANDBOX_DIR}")
